In [1]:
!pip install -q sentence-transformers chromadb langchain langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 k

In [2]:
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [3]:
documents = [
    """Refund Policy:
    Customers can request a refund within 30 days of purchase.
    Products must be unused and in original packaging.""",

    """Shipping Policy:
    Orders are shipped within 3-5 business days.
    International shipping may take longer.""",

    """Support Policy:
    Customer support is available 24/7 via email and chat."""
]

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = []
for doc in documents:
    chunks.extend(text_splitter.split_text(doc))

print(f"Total Chunks Created: {len(chunks)}")

Total Chunks Created: 3


In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_454/2127729888.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
vector_db = Chroma.from_texts(
    texts=chunks,
    embedding=embedding_model,
    persist_directory="chroma_store"
)

print("Vector DB created successfully")

Vector DB created successfully


In [46]:
query = "What is the the environmental factors of Anxiety?"

results = vector_db.similarity_search(
    query=query,
    k=1
)

print(results)

[Document(id='2da4eebf-0c75-4620-b270-20d1c4c0271e', metadata={'producer': 'PDF Candy', 'creator': 'PDF Candy', 'creationdate': '2026-03-23T14:29:16+00:00', 'title': '', 'moddate': '2026-03-23T14:29:16+00:00', 'source': '/content/AyurGenixAI_Dataset.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Tests\nModerate\nto\tSevere\nVaries,\ndepends\ton\nseverity\nStress,\tTrauma,\nDepression,\tGenetic\nfactors\nAntidepressants,\nBenzodiazepines\nStress,\tGenetics,\nTrauma\nOverwork,\tPersonal\nissues\nPoor\nSleep\nHigh\nStress\nLow\tto\nModerate\nFamily\tHistory\tof\nAnxiety\nBalanced\tdiet,\nLow\tcaffeine\nNone\nAll\nseasons\n20-40\tyears\nMostly\nFemale\nHigh-stress\tjob\nFavors\tholistic\npractices\nLavender\tOil,\nChamomile\tTea\nAshwagandha,\nBrahmi\nJatamansi\n(3g),\tWarm\nwater\t(200ml)\nVata\nVata-Prakriti\nAvoid\tcaffeine\npractice\nmindfulness')]


In [47]:
!pip install -q sentence-transformers faiss-cpu langchain langchain-community groq pypdf langchain_groq

In [49]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/AyurGenixAI_Dataset.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages")

Loaded 9 pages


In [50]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")

Total chunks created: 821


In [51]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
from langchain_community.vectorstores import FAISS
import faiss

dim = 384


vector_db = FAISS.from_documents(chunks, embeddings)
print("FAISS vector store created from documents.")

FAISS vector store created from documents.


In [53]:
import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = " "

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [55]:
from langchain_core.prompts import PromptTemplate

query = "What is the the diagonosis of Adrenal Insufficiency?"

docs = vector_db.similarity_search(query, k=5)

context = "\n\n".join([d.page_content for d in docs])

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful assistant.
Use ONLY the context below to answer.

Question:
{question}

Context:
{context}

If the answer is not in the context, say "I don't know".
"""
)

final_prompt = prompt.format(
    context=context,
    question=query
)

response = llm.invoke(final_prompt)
print(response.content)

The diagnosis of Adrenal Insufficiency is:

1. Blood Tests, ACTH Stimulation Test
2. Fatigue, low blood pressure, dizziness
3. Autoimmune diseases, Infections
4. Irregular sleep
5. High stress
6. Low Family history of adrenal issues
7. Low-sodium, Balanced diet
8. Urinary Cortisol Test
